# 🗃️ SQL Analytics — TikTok & Instagram @aroaxinping

Análisis de métricas de redes sociales usando **SQL puro** sobre una base de datos SQLite.

Período: 24 Feb – 24 Mar 2026  
Tablas: `tiktok_videos`, `tiktok_overview`, `instagram_reels`, `instagram_daily`

---

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DB = Path('..') / 'social_media.db'
conn = sqlite3.connect(DB)

def sql(query, title=None):
    df = pd.read_sql_query(query, conn)
    if title:
        print(f'\n### {title}')
    return df

print('✅ Conectado a social_media.db')
for t in ['tiktok_videos','tiktok_overview','instagram_reels','instagram_daily']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0,0]
    print(f'  {t}: {n} filas')

✅ Conectado a social_media.db
  tiktok_videos: 10 filas
  tiktok_overview: 6 filas
  instagram_reels: 30 filas
  instagram_daily: 28 filas


## 1. Exploración básica

In [2]:
sql("""
SELECT
    COUNT(*) AS total_videos,
    SUM(views) AS total_vistas,
    ROUND(AVG(engagement_rate_pct), 2) AS er_medio,
    ROUND(AVG(completion_pct), 2) AS completion_medio,
    SUM(new_followers) AS seguidores_ganados,
    MAX(views) AS video_mas_viral
FROM tiktok_videos
""", 'KPIs globales TikTok')


### KPIs globales TikTok


,total_videos,total_vistas,er_medio,completion_medio,seguidores_ganados,video_mas_viral
0,10,2592000,18.48,36.52,2536,777000


In [3]:
sql("""
SELECT
    COUNT(*) AS total_reels,
    SUM(visualizaciones) AS total_vistas,
    ROUND(AVG(engagement_rate), 2) AS er_medio,
    ROUND(AVG(save_rate), 2) AS save_rate_medio,
    SUM(seguidores_ganados) AS seguidores_ganados
FROM instagram_reels
""", 'KPIs globales Instagram')


### KPIs globales Instagram


,total_reels,total_vistas,er_medio,save_rate_medio,seguidores_ganados
0,30,7269352,12.19,0.94,7682


In [4]:
sql("""
SELECT tema, COUNT(*) AS reels,
    ROUND(AVG(engagement_rate), 2) AS er_medio,
    ROUND(AVG(save_rate), 2) AS save_rate_medio,
    SUM(seguidores_ganados) AS seguidores
FROM instagram_reels
GROUP BY tema HAVING COUNT(*) >= 2
ORDER BY er_medio DESC
""", 'Instagram — Top temáticas por engagement')


### Instagram — Top temáticas por engagement


,tema,reels,er_medio,save_rate_medio,seguidores
0,Terminal/Bash,2,13.10,2.10,505
1,Tech humor,22,12.20,0.86,3288
2,SQL,2,11.68,0.96,761


In [5]:
sql("""
SELECT franja, COUNT(*) AS reels,
    ROUND(AVG(engagement_rate), 2) AS er_medio,
    ROUND(AVG(save_rate), 2) AS save_rate_medio
FROM instagram_reels
GROUP BY franja
ORDER BY er_medio DESC
""", 'Instagram — Mejor franja horaria para publicar')


### Instagram — Mejor franja horaria para publicar


,franja,reels,er_medio,save_rate_medio
0,Mañana (6-12h),27,12.53,0.90
1,Tarde (12-18h),2,10.95,1.82
2,Madrugada (0-6h),1,5.48,0.39


## 2. Window Functions

In [6]:
sql("""
SELECT
    title,
    views,
    engagement_rate_pct,
    RANK() OVER (ORDER BY views DESC) AS rank_vistas,
    RANK() OVER (ORDER BY engagement_rate_pct DESC) AS rank_er,
    views - LAG(views) OVER (ORDER BY views DESC) AS gap_respecto_anterior
FROM tiktok_videos
ORDER BY rank_vistas
""", 'TikTok — Ranking con LAG (gap entre vídeos)')


### TikTok — Ranking con LAG (gap entre vídeos)


,title,views,engagement_rate_pct,rank_vistas,rank_er,gap_respecto_anterior
0,te amo mi chico informático,777000,12.14,1,9,NaN
1,Te las mereces 💐,426000,26.18,2,1,-351000.00
2,xd #informatica #humor,379000,20.39,3,4,-47000.00
3,me ayudan? #geforcertx,299000,11.60,4,10,-80000.00
4,Cosorro #poo,195000,25.52,5,2,-104000.00
5,20 años con 20 años de experiencia,142000,15.12,6,7,-53000.00
6,una <div>a mi terminal,104000,17.96,7,6,-38000.00
7,lo q mas duele en una ruptura,100000,19.04,8,5,-4000.00
8,la espaldaaaa,95000,22.05,9,3,-5000.00
9,#datascience #cienciadedatos,75000,14.80,10,8,-20000.00


In [7]:
sql("""
SELECT
    fecha, seguidores_ganados,
    SUM(seguidores_ganados) OVER (ORDER BY fecha) AS seguidores_acumulados,
    ROUND(AVG(seguidores_ganados) OVER (
        ORDER BY fecha ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 1) AS media_movil_7d
FROM instagram_daily
ORDER BY fecha
""", 'Instagram — Running total + media móvil 7 días')


### Instagram — Running total + media móvil 7 días


,fecha,seguidores_ganados,seguidores_acumulados,media_movil_7d
0,2026-02-24,0,0,0.00
1,2026-02-25,1,1,0.50
2,2026-02-26,0,1,0.30
3,2026-02-27,2,3,0.80
4,2026-02-28,5,8,1.60
5,2026-03-01,25,33,5.50
6,2026-03-02,43,76,10.90
7,2026-03-03,99,175,25.00
8,2026-03-04,89,264,37.60
9,2026-03-05,140,404,57.60


In [8]:
sql("""
SELECT * FROM (
    SELECT tema, descripcion_corta, fecha,
        engagement_rate, save_rate,
        ROW_NUMBER() OVER (PARTITION BY tema ORDER BY engagement_rate DESC) AS rank_en_tema
    FROM instagram_reels
)
WHERE rank_en_tema <= 3
ORDER BY tema, rank_en_tema
""", 'Instagram — Top 3 reels por temática (PARTITION BY)')


### Instagram — Top 3 reels por temática (PARTITION BY)


,tema,descripcion_corta,fecha,engagement_rate,save_rate,rank_en_tema
0,Buenas prácticas,muy necesario el README\n#techhumor #programac...,2026-03-19,15.52,1.04,1
1,Humor personal,te amo mi chico informático \n#programador #pa...,2026-02-28,2.63,0.31,1
2,Programación general,preparadisimo para el mundo laboral #leetcode ...,2026-03-03,13.46,0.79,1
3,Python,explicame 🐍 #programador #python #techhumor,2026-03-14,16.14,1.02,1
4,SQL,tú sabes??\n#sql #techhumor #programacion #inf...,2026-03-22,12.08,1.08,1
5,SQL,Excel -ente SELECT \n#Sql #sqldeveloper #techumor,2026-03-11,11.29,0.83,2
6,Tech humor,la espaldaaaa \n#techhumor #programacion #info...,2026-03-20,18.02,0.91,1
7,Tech humor,php siempre funciona 💤 gracias @mouredev #tech...,2026-03-17,17.86,1.03,2
8,Tech humor,alguien me ayuda?? @nvidiageforcees 😭\n#techhu...,2026-03-16,17.84,0.45,3
9,Terminal/Bash,una <div>a mi terminal #womenintech \n#techgir...,2026-03-05,15.58,1.40,1


## 3. CTEs y Subqueries

In [9]:
sql("""
WITH medias AS (
    SELECT AVG(engagement_rate) AS avg_er,
           AVG(save_rate) AS avg_save,
           AVG(share_rate) AS avg_share
    FROM instagram_reels
)
SELECT r.fecha, r.descripcion_corta, r.engagement_rate, r.save_rate, r.share_rate, r.tema
FROM instagram_reels r
CROSS JOIN medias m
WHERE r.engagement_rate > m.avg_er
  AND r.save_rate > m.avg_save
  AND r.share_rate > m.avg_share
ORDER BY r.engagement_rate DESC
""", 'CTE — Reels que superan la media en ER + save + share a la vez')


### CTE — Reels que superan la media en ER + save + share a la vez


,fecha,descripcion_corta,engagement_rate,save_rate,share_rate,tema
0,2026-03-17,php siempre funciona 💤 gracias @mouredev #tech...,17.86,1.03,1.58,Tech humor
1,2026-03-23,evento canónico \n#informatica #techhumor #pro...,17.44,1.16,1.35,Tech humor
2,2026-03-14,explicame 🐍 #programador #python #techhumor,16.14,1.02,4.09,Python
3,2026-03-19,muy necesario el README\n#techhumor #programac...,15.52,1.04,1.91,Buenas prácticas
4,2026-03-20,recursión emocional // aropoemas\n#poemastech ...,14.98,1.59,1.91,Tech humor
5,2026-03-18,dedícaselo a tu https ✉️\n#techhumor #programa...,13.67,1.47,1.93,Tech humor


In [10]:
sql("""
SELECT r.fecha, r.tema, r.descripcion_corta, r.engagement_rate,
    ROUND((
        SELECT AVG(r2.engagement_rate)
        FROM instagram_reels r2 WHERE r2.tema = r.tema
    ), 2) AS media_er_tema,
    ROUND(r.engagement_rate - (
        SELECT AVG(r2.engagement_rate)
        FROM instagram_reels r2 WHERE r2.tema = r.tema
    ), 2) AS delta_vs_tema
FROM instagram_reels r
WHERE r.engagement_rate > (
    SELECT AVG(r2.engagement_rate) FROM instagram_reels r2 WHERE r2.tema = r.tema
)
ORDER BY delta_vs_tema DESC
""", 'Subquery correlacionada — Reels por encima de la media de su temática')


### Subquery correlacionada — Reels por encima de la media de su temática


,fecha,tema,descripcion_corta,engagement_rate,media_er_tema,delta_vs_tema
0,2026-03-20,Tech humor,la espaldaaaa \n#techhumor #programacion #info...,18.02,12.20,5.82
1,2026-03-17,Tech humor,php siempre funciona 💤 gracias @mouredev #tech...,17.86,12.20,5.66
2,2026-03-16,Tech humor,alguien me ayuda?? @nvidiageforcees 😭\n#techhu...,17.84,12.20,5.64
3,2026-02-27,Tech humor,lo siento profe \n#programacion #techhumor #in...,17.50,12.20,5.30
4,2026-03-23,Tech humor,evento canónico \n#informatica #techhumor #pro...,17.44,12.20,5.24
5,2026-03-20,Tech humor,recursión emocional // aropoemas\n#poemastech ...,14.98,12.20,2.78
6,2026-03-05,Terminal/Bash,una <div>a mi terminal #womenintech \n#techgir...,15.58,13.10,2.48
7,2026-03-11,Tech humor,lo más difícil #programacion #informatica #tec...,14.43,12.20,2.23
8,2026-03-22,Tech humor,no te confundas mi ciela\n#techhumor #informat...,14.43,12.20,2.23
9,2026-03-14,Tech humor,no hagas force push\n#programacion #developer ...,13.76,12.20,1.56


## 4. Análisis Cross-Platform

In [11]:
sql("""
SELECT plataforma, COUNT(*) AS total_posts,
    ROUND(AVG(visualizaciones), 0) AS vistas_medias,
    ROUND(AVG(engagement_rate), 2) AS er_medio,
    ROUND(AVG(save_rate), 2) AS save_rate_medio,
    ROUND(AVG(share_rate), 2) AS share_rate_medio,
    SUM(seguidores_ganados) AS seguidores_totales,
    ROUND(CAST(SUM(seguidores_ganados) AS FLOAT) / NULLIF(SUM(visualizaciones),0) * 1000, 4) AS follower_rate_1k
FROM (
    SELECT 'TikTok' AS plataforma, views AS visualizaciones,
           engagement_rate_pct AS engagement_rate, save_rate_pct AS save_rate,
           share_rate_pct AS share_rate, new_followers AS seguidores_ganados
    FROM tiktok_videos
    UNION ALL
    SELECT 'Instagram', visualizaciones, engagement_rate,
           save_rate, share_rate, seguidores_ganados
    FROM instagram_reels
)
GROUP BY plataforma
""", 'UNION — KPIs comparativos TikTok vs Instagram')


### UNION — KPIs comparativos TikTok vs Instagram


,plataforma,total_posts,vistas_medias,er_medio,save_rate_medio,share_rate_medio,seguidores_totales,follower_rate_1k
0,Instagram,30,242312.00,12.19,0.94,1.26,7682,1.06
1,TikTok,10,259200.00,18.48,1.24,1.91,2536,0.98


In [12]:
sql("""
SELECT plataforma, dia_semana, posts,
    ROUND(er_medio, 2) AS er_medio,
    RANK() OVER (PARTITION BY plataforma ORDER BY er_medio DESC) AS rank_dia
FROM (
    SELECT 'TikTok' AS plataforma,
           CASE CAST(strftime('%w', published_date) AS INT)
               WHEN 0 THEN 'Sunday' WHEN 1 THEN 'Monday' WHEN 2 THEN 'Tuesday'
               WHEN 3 THEN 'Wednesday' WHEN 4 THEN 'Thursday'
               WHEN 5 THEN 'Friday' ELSE 'Saturday'
           END AS dia_semana,
           COUNT(*) AS posts, AVG(engagement_rate_pct) AS er_medio
    FROM tiktok_videos GROUP BY dia_semana
    UNION ALL
    SELECT 'Instagram', dia_semana, COUNT(*), AVG(engagement_rate)
    FROM instagram_reels GROUP BY dia_semana
)
ORDER BY plataforma, rank_dia
""", 'Cross-platform — Mejor día por plataforma (UNION + WINDOW)')


### Cross-platform — Mejor día por plataforma (UNION + WINDOW)


,plataforma,dia_semana,posts,er_medio,rank_dia
0,Instagram,Tuesday,3,14.86,1
1,Instagram,Monday,4,14.71,2
2,Instagram,Thursday,3,13.37,3
3,Instagram,Friday,5,13.09,4
4,Instagram,Wednesday,4,12.43,5
5,Instagram,Sunday,4,10.68,6
6,Instagram,Saturday,7,9.20,7
7,TikTok,Sunday,1,25.52,1
8,TikTok,Friday,1,20.39,2
9,TikTok,Saturday,2,19.16,3


In [13]:
conn.close()
print('✅ Análisis completado')

✅ Análisis completado
